In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant


CSV_PATH = Path(r"E:\NO-NC\normalization\correlation_20261019.csv")
OUTPUT_PATH = CSV_PATH.parent / "VIF_20261019.csv"

# 读取数据并清理列名
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
df.columns = df.columns.astype(str).str.strip()

# 将 NoData 标记和无穷值转为缺失值，所有列按连续数值变量处理
df = df.replace([-9999, -9999.0, np.inf, -np.inf], np.nan)
df = df.apply(pd.to_numeric, errors="coerce")

# 删除全缺失列和常数列
all_missing = df.columns[df.isna().all()].tolist()
constant = [
    col for col in df.columns
    if df[col].nunique(dropna=True) <= 1
]
drop_cols = sorted(set(all_missing + constant))
X = df.drop(columns=drop_cols).copy()

if X.shape[1] < 2:
    raise ValueError("计算 VIF 至少需要两个有效解释变量。")

# 找出在参与 VIF 计算的变量中含缺失值的记录
missing_mask = X.isna().any(axis=1)
dropped_records = df.loc[missing_mask].copy()

print(f"原始样本记录数：{len(df)}")
print(f"因缺失值将删除的记录数：{len(dropped_records)}")
print(f"删除的全缺失或常数变量：{drop_cols}")

# 打印所有将被删除的记录
if not dropped_records.empty:
    print("\n以下记录因解释变量存在缺失值，将不参与 VIF 计算：")

    # CSV 第 1 行是表头，因此 pandas 索引 + 2 对应 CSV 实际行号
    records_to_print = dropped_records.copy()
    records_to_print.insert(
        0,
        "CSV_data_row_number",
        dropped_records.index + 2
    )

    print(records_to_print.to_string(index=False))
else:
    print("\n没有记录因缺失值被删除。")

# 删除含缺失值的记录
X = X.loc[~missing_mask].copy()

if X.empty:
    raise ValueError("删除缺失记录后没有可用于计算 VIF 的样本。")

print(f"\n实际用于计算 VIF 的样本数：{len(X)}")
print(f"参与计算的解释变量数：{X.shape[1]}")

# 添加截距项并计算 VIF
X_with_constant = add_constant(X, has_constant="add")
X_array = X_with_constant.to_numpy(dtype=float)

results = []
for i, variable in enumerate(X_with_constant.columns):
    if variable == "const":
        continue

    vif = variance_inflation_factor(X_array, i)
    results.append({
        "variable": variable,
        "VIF": vif,
        "VIF_gt_5": bool(vif > 5),
    })

vif_table = (
    pd.DataFrame(results)
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

vif_table.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("\nVIF 结果：")
print(vif_table.to_string(index=False))
print(f"\n结果已保存至：{OUTPUT_PATH}")

原始样本记录数：4038
因缺失值将删除的记录数：2
删除的全缺失或常数变量：[]

以下记录因解释变量存在缺失值，将不参与 VIF 计算：
 CSV_data_row_number  Minimum temperature  Maximum temperature  Average temperature  Precipitation  Solar radiation  Vapor pressure  Wind speed  Climate classification      DEM    Slope   Aspect    TWI  Longitude  Latitude       pH      CEC  Texture  Parent material  G_As_05  G_As_10  G_As_50       Ag       As       Au       Cu      Hg       Pb       Sn      Zn  Land use  Gravitational field
                3634                  NaN                  NaN                  NaN            NaN              NaN             NaN         NaN                     NaN 0.282321 0.046904 0.955764 0.4512   0.908778  0.601118 0.824427 0.153802 0.454545              0.0 0.891109 0.216151 0.048705 0.430140 0.071992 0.435992 0.229937 0.05062 0.301634 0.084922 0.45263    0.5625             0.822606
                3877                  NaN                  NaN                  NaN            NaN              NaN             NaN        

C:\ProgramData\Anaconda3\lib\site-packages\statsmodels\tsa\tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only.
  x = pd.concat(x[::order], 1)



VIF 结果：
              variable        VIF  VIF_gt_5
   Average temperature 246.871684      True
   Gravitational field 125.840247      True
                   DEM  89.188607      True
              Latitude  87.868609      True
   Minimum temperature  86.648465      True
   Maximum temperature  68.921961      True
             Longitude  28.322846      True
        Vapor pressure  20.274816      True
            Wind speed   9.915574      True
       Solar radiation   8.265598      True
                    pH   6.667967      True
                   CEC   5.552450      True
         Precipitation   4.905606     False
                    Hg   4.279613     False
               G_As_10   4.013529     False
                    As   3.059706     False
               G_As_50   3.018868     False
               G_As_05   2.956643     False
                    Zn   2.887708     False
               Texture   2.319526     False
                    Sn   2.298286     False
Climate classification 